# Twitter Sentiment Analysis (30 points)


**Import the notebook into Colab and work inside it!**

**Contestant's name:**


John Smart suddenly registered on Twitter on a whim. He hasn't discovered emojis yet, so he can only infer the mood of a tweet from the text.
Help him determine what sentiment the messages were written in!

---
# Preparations

**Guides to tools needed for solving the task:**
1. [Pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
2. [Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
3. [Tokenization](https://medium.com/@utkarsh.kant/tokenization-a-complete-guide-3f2dd56c0682)
4. [Tokenization, Mapping and Padding](https://medium.com/@lokaregns/preparing-text-data-for-transformers-tokenization-mapping-and-padding-9fbfbce28028)
5. [PyTorch Training/Inference](https://pytorch.org/tutorials/beginner/introyt/trainingyt.html)
6. [PyTorch Datasets and DataLoaders](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html)
7. [Pytorch NN Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
8. [TorchText Tokenizer](https://pytorch.org/text/stable/data_utils.html)
9. [Stop Word](https://en.wikipedia.org/wiki/Stop_word)
10. [Logistic Regression](https://www.spiceworks.com/tech/artificial-intelligence/articles/what-is-logistic-regression/#:~:text=Logistic%20regression%20is%20a%20supervised%20machine%20learning%20algorithm%20that%20accomplishes,1%2C%20or%20true%2Ffalse.)
11. [Logistic Regression sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)

The **Tweets** needed for the solution come from the [Sentiment140](https://www.kaggle.com/datasets/kazanova/sentiment140) dataset (1.6M tweets, Stanford).

Download: [Kaggle Sentiment140](https://www.kaggle.com/datasets/kazanova/sentiment140)

In [ ]:
# @title Install Dependencies
!pip install pandas --quiet
!pip install torchtext --quiet

In [ ]:
# @title Download Tweet Dataset (Sentiment140)
# Original Google Drive link is no longer available.
# Alternative: download from Kaggle:
# https://www.kaggle.com/datasets/kazanova/sentiment140

# With Kaggle CLI:
# !pip install -q kaggle
# !kaggle datasets download -d kazanova/sentiment140
# !unzip -qq sentiment140.zip

# Original (not available):
# !gdown -qq 1penca66caOrgagaQvrKCBDrCN_g1ZN35
# !unzip -qq twitter.zip -d .
# !rm -rf twitter.zip

In [ ]:
# !pip install -q kaggle
# !kaggle datasets download -d kazanova/sentiment140
# !unzip -qq sentiment140.zip


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Dataset URL: https://www.kaggle.com/datasets/kazanova/sentiment140
License(s): other




  0%|          | 0.00/80.9M [00:00<?, ?B/s]
  1%|          | 1.00M/80.9M [00:00<01:01, 1.37MB/s]
  2%|▏         | 2.00M/80.9M [00:00<00:30, 2.68MB/s]
  4%|▎         | 3.00M/80.9M [00:01<00:20, 3.94MB/s]
  5%|▍         | 4.00M/80.9M [00:01<00:15, 5.05MB/s]
  6%|▌         | 5.00M/80.9M [00:01<00:13, 6.01MB/s]
  7%|▋         | 6.00M/80.9M [00:01<00:11, 6.75MB/s]
  9%|▊         | 7.00M/80.9M [00:01<00:10, 7.35MB/s]
 10%|▉         | 8.00M/80.9M [00:01<00:09, 7.81MB/s]
 11%|█         | 9.00M/80.9M [00:01<00:09, 8.19MB/s]
 12%|█▏        | 10.0M/80.9M [00:01<00:08, 8.50MB/s]
 14%|█▎        | 11.0M/80.9M [00:01<00:08, 8.52MB/s]
 15%|█▍        | 12.0M/80.9M [00:02<00:08, 8.58MB/s]
 16%|█▌        | 13.0M/80.9M [00:02<00:08, 8.65MB/s]
 17%|█▋        | 14.0M/80.9M [00:02<00:08, 8.63MB/s]
 19%|█▊        | 15.0M/80.9M [00:02<00:07, 8.76MB/s]
 20%|█▉        | 16.0M/80.9M [00:02<00:07, 8.58MB/s]
 21%|██        | 17.0M/80.9M [00:02<00:07, 8.90MB/s]
 22%|██▏       | 18.0M/80.9M [00:02<00:07, 8.92MB/s]
 

## Required Libraries

We have imported some libraries to get started, but feel free to use any PyTorch-based tools if needed. Please note that Keras and TensorFlow are **NOT ALLOWED** for solving this task!

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.utils import shuffle
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

## Visualizing the Twitter Dataset

The following code visualizes the size of the dataset and the first five Tweets.

Each text also has a classification indicating whether the given Tweet is **positive** (4) or **negative** (0).

[Pandas Dataframe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)

In [4]:
header_list = ["target", "id", "date", "query", "user", "text"]
df = pd.read_csv('./sentiment140/training.1600000.processed.noemoticon.csv',
                 encoding = "ISO-8859-1", names=header_list)
print("Number of Tweets:", len(df))
df.head()

Number of Tweets: 1600000


,target,id,date,query,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


# Task 1: Exploratory Data Analysis (14 points)

In the first task, we want to perform initial analysis and data cleaning on a Twitter dataset to better understand the Tweets.

1. Transform the 'target' field so that instead of '0' and '4', '**0**' and '**1**' values appear for **negative** and **positive** sentiment tweets! (binary encoding) (1 point)

2. Split the dataset into **test** and **training** data! The split should be **20-80%**. The order of the data should not change during processing! (2 points)

3. Print **5-5** negative and positive sentiment tweet pairs from the **training dataset**! (1 point)

4. Tokenize every tweet in both the **training and test datasets**! Use torchtext's **get_tokenizer** function to tokenize the tweets! [TorchText Tokenizer](https://pytorch.org/text/stable/data_utils.html) (2 points)

5. Calculate how many unique tokens the **training dataset** contains! (1 point)

6. Visualize with a bar plot the 100 most frequently used words and their occurrence counts in the **training dataset**! (2 points)

7. Visualize the 100 least frequently used words and their occurrence counts in the **training dataset**! (2 points)

In [8]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')

def target_binary_encode(df):
    return df['target'].map({0:0, 4:1})

def test_train_split(df):
    l = len(df)
    return df.iloc[:l*0.8], df[l*0.8:]

def printing_five(df):
    print(df[df['target'] == 0].head(5))
    print(df[df['target'] == 1].head(5))

def tokenize_tweets(train_df, test_df):
    return train_df['text'].apply(word_tokenize), test_df['text'].apply(word_tokenize)

def unique_token_count(df):
    all_tokens = [token.lower() for text_tokens in df['text'] for token in text_tokens]
    return len(set(all_tokens))

def top_hundred_words(df):
    all_tokens = [token.lower() for text_tokens in df['text'] for token in text_tokens]
    counter = Counter(all_tokens)

    most_common_100 = counter.most_common(100)

    words, counts = zip(*most_common_100)

    plt.figure(figsize=(18, 5))
    plt.bar(words, counts)
    plt.xticks(rotation=90)
    plt.title("Top 100 Most Frequent Words (Training Set)")
    plt.show()

def bottom_hundred_words():
    all_tokens = [token.lower() for text_tokens in df['text'] for token in text_tokens]
    counter = Counter(all_tokens)

    most_common_100 = counter.most_common()[-101:-1]

    words, counts = zip(*most_common_100)

    plt.figure(figsize=(18, 5))
    plt.bar(words, counts)
    plt.xticks(rotation=90)
    plt.title("Top 100 Most Frequent Words (Training Set)")
    plt.show()

AttributeError: module 'nltk' has no attribute 'internals'

# Task 2: Data Cleaning (10 points)

As you can see from the above results, the number of unique words is very high. This is possible because the training tweets contain many punctuation marks, stop words [Stop Word](https://en.wikipedia.org/wiki/Stop_word), multiple inflected forms of words, and very rare words, even different user IDs. (Stop words are the most frequently used words in a given language, for example in English they can be words like "a", "an", "and", "but", "with", or various forms of the verb "to be".)

Your tasks are as follows:

1. Remove **punctuation marks** from the training dataset! (1 point)

2. Remove **stop words** from the training dataset! (2 points)

3. Remove all words from the training dataset that **appear only once**! (2 points)

4. Remove all **user mentions** from the training dataset. (@USER_ID) (4 points)

5. Retokenize both the training and test datasets. (1 point)

In [ ]:
def clean_dataset():
    raise NotImplementedError("This function has not been implemented yet.")

def retokenize_dataset():
    raise NotImplementedError("This function has not been implemented yet.")

# Task 3: Training a Classifier for Tweet Classification (6 points)

The final task is to perform a logistic regression task. Logistic regression is a multivariate method that allows us to categorize cases according to the categories of the dependent variable. [Logistic Regression](https://www.spiceworks.com/tech/artificial-intelligence/articles/what-is-logistic-regression/#:~:text=Logistic%20regression%20is%20a%20supervised%20machine%20learning%20algorithm%20that%20accomplishes,1%2C%20or%20true%2Ffalse.)

1. Using the sklearn package, use LogisticRegression with "saga" solver and apply it to the **training dataset**! [Logistic Regression sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) (2 points)

2. Using the predict function, test the model's performance on the **test set**! (Don't forget to **tokenize** it as well!) `LogisticRegression.predict()` (4 points)

Training the above classifier is **time-consuming** (a few minutes), so we recommend that while the model is training, you start working on the other tasks.

In [ ]:
def train_classifier():
    raise NotImplementedError("This function has not been implemented yet.")


---
**You have reached the end of the task**. Download the finished Notebook as follows:
```
File → Download → Download .ipynb
```
then upload it together with the **other solutions** packaged as **.zip** into the CMS.